# 01 — Filters and Pipelines

This notebook introduces the two foundational concepts in Soma:

- **Filter**: The basic computation unit. It has two phases:
  - `fit(x, y)` — learn state from training data (returns a dict)
  - `forward(x, state)` — transform data using that learned state
- **Pipeline**: A chain of filters that executes sequentially.

> **Prerequisites**: Install soma with `cd soma-python && maturin develop`

## 1.1 — Your first Filter

A Filter is a Python class that inherits from `soma.Filter`. The simplest
filter just transforms data without learning anything (stateless).

In [ ]:
from soma import Filter, Pipeline

class Doubler(Filter):
    """A stateless filter that doubles every value."""

    def forward(self, x, state):
        return [v * 2 for v in x]

# Instantiate and test directly
f = Doubler()
state = f.fit([1, 2, 3])       # no-op for stateless filters
result = f.forward([1, 2, 3], state)

print(f"fit() returns:    {state}")
print(f"forward() returns: {result}")

## 1.2 — A trainable Filter (with state)

Most useful filters **learn something** from training data. The key rule:

- `fit()` returns a dict with the learned state
- `forward()` receives that state as an argument — it does **not** live on `self`

This separation is what makes caching possible: same config + same data = same state.

In [ ]:
class Normalizer(Filter):
    """Learns mean and std from data, then normalizes."""

    def fit(self, x, y=None):
        mean = sum(x) / len(x)
        std = (sum((v - mean) ** 2 for v in x) / len(x)) ** 0.5
        # State is a dict — it will be passed to forward()
        return {"mean": mean, "std": std}

    def forward(self, x, state):
        mean = state["mean"]
        std = state["std"]
        if std == 0:
            return [0.0] * len(x)
        return [(v - mean) / std for v in x]

norm = Normalizer()
state = norm.fit([10.0, 20.0, 30.0])
print(f"Learned state: {state}")

result = norm.forward([10.0, 20.0, 30.0], state)
print(f"Normalized:    {result}")

# The same state can be applied to new data
new_data = norm.forward([15.0, 25.0], state)
print(f"New data:      {new_data}")

## 1.3 — Filters with parameters

Filters can have **parameters** (constructor args) that affect their behavior.
Parameters are part of the config hash — changing them creates a new cache key.

Private attributes (prefixed with `_`) are excluded from config hashing.

In [ ]:
class PolynomialFeatures(Filter):
    """Expands input features to polynomial terms."""

    def __init__(self, degree=2):
        super().__init__(degree=degree)
        # degree is a parameter (public) — included in config hash
        # _log is internal (private) — excluded from config hash
        self._log = []

    def fit(self, x, y=None):
        self._log.append("fitted")
        return {}  # stateless transform, but could learn e.g. scaling factors

    def forward(self, x, state):
        # x^1, x^2, ..., x^degree for each value
        return [[v ** p for p in range(1, self.degree + 1)] for v in x]

poly = PolynomialFeatures(degree=3)
print(f"Parameters:  degree={poly.degree}")
print(f"Internal:    _log={poly._log}")

poly.fit([1.0, 2.0])
result = poly.forward([2.0, 3.0], {})
print(f"Poly(2.0):   {result[0]}")  # [2, 4, 8]
print(f"Poly(3.0):   {result[1]}")  # [3, 9, 27]

## 1.4 — Building a Pipeline

A `Pipeline` chains filters together. When you call `pipeline.fit(x)`:

1. Filter 1 fits on `x`, produces `state_1`, then `forward(x, state_1)` → `x_2`
2. Filter 2 fits on `x_2`, produces `state_2`, then `forward(x_2, state_2)` → `x_3`
3. ... and so on

When you call `pipeline.predict(x)`, it only runs `forward()` using the saved states.

In [ ]:
# A pipeline that normalizes, then doubles
pipeline = Pipeline([Normalizer(), Doubler()])

print(f"Filters:   {pipeline.filter_names()}")
print(f"Is fitted: {pipeline.is_fitted()}")

# Fit on training data
pipeline.fit([10.0, 20.0, 30.0, 40.0, 50.0])
print(f"Is fitted: {pipeline.is_fitted()}")

# Predict on new data
result = pipeline.predict([15.0, 25.0, 35.0])
print(f"Result:    {result}")
# 15 normalized → (15-30)/~14.14 ≈ -1.06, then doubled ≈ -2.12

## 1.5 — Named filters in pipelines

Filters get auto-named from their class (snake_case), but you can override with tuples.

In [ ]:
# Auto-naming: CamelCase → snake_case
auto = Pipeline([Normalizer(), Doubler()])
print(f"Auto names: {auto.filter_names()}")  # ['normalizer', 'doubler']

# Explicit naming with tuples
named = Pipeline([
    ("preprocess", Normalizer()),
    ("amplify", Doubler()),
])
print(f"Custom names: {named.filter_names()}")  # ['preprocess', 'amplify']

## 1.6 — Filter metadata

Filters can declare metadata that affects how the runtime handles them.
These are set as class attributes prefixed with `_`.

In [ ]:
class StatelessDoubler(Filter):
    """A filter that declares itself as stateless and non-cacheable."""
    _kind = "stateless"       # no fit() phase needed
    _cacheable = False        # don't cache outputs
    _differentiable = False   # gradients don't flow through
    _stream_mode = "fixed"    # state doesn't change during streaming

    def forward(self, x, state):
        return [v * 2 for v in x]

class EvolvingAccumulator(Filter):
    """A streaming filter whose state evolves with each chunk."""
    _kind = "trainable"
    _stream_mode = "evolving"  # state updates during streaming

    def fit(self, x, y=None):
        return {"running_sum": sum(x)}

    def forward(self, x, state):
        return [v + state["running_sum"] for v in x]

# These metadata attributes are read by the Rust runtime
# to optimize execution, caching, and streaming behavior
print("StatelessDoubler metadata:")
print(f"  _kind:           {StatelessDoubler._kind}")
print(f"  _cacheable:      {StatelessDoubler._cacheable}")
print(f"  _stream_mode:    {StatelessDoubler._stream_mode}")
print()
print("EvolvingAccumulator metadata:")
print(f"  _kind:           {EvolvingAccumulator._kind}")
print(f"  _stream_mode:    {EvolvingAccumulator._stream_mode}")

## 1.7 — Data types

Soma accepts several data formats. The Rust runtime converts them automatically.

| Python input | Soma Value | Python output |
|---|---|---|
| `[1.0, 2.0, 3.0]` | 1D Tensor | `[1.0, 2.0, 3.0]` |
| `[[1.0, 2.0], [3.0, 4.0]]` | 2D Tensor | `[[1.0, 2.0], [3.0, 4.0]]` |
| `{"key": "value"}` | JSON | `{"key": "value"}` |

In [ ]:
class Identity(Filter):
    """Passes data through unchanged — useful for testing data conversion."""
    def forward(self, x, state):
        return x

identity = Pipeline([Identity()])
identity.fit([1.0])

# 1D tensor
print("1D:", identity.predict([1.0, 2.0, 3.0]))

# 2D tensor
print("2D:", identity.predict([[1.0, 2.0], [3.0, 4.0]]))

# JSON / dict
print("Dict:", identity.predict({"name": "soma", "version": 1}))

## 1.8 — Error handling

Soma provides clear errors when things go wrong.

In [ ]:
# Error: predict before fit
unfitted = Pipeline([Doubler()])
try:
    unfitted.predict([1.0, 2.0])
except RuntimeError as e:
    print(f"Expected error: {e}")

---

**Next:** [02 — Caching and State](./02_caching_and_state.ipynb) — how Soma caches fit states and forward outputs.